# P1KS development window (2015–2019 only)

Calibration for the power check of the planned Korea short-sale test. **It reads nothing dated 2020 or later**, and
the code stops if any date reaches 2020. It prints summary statistics across stocks only.

**Before running:** in Colab, open the key icon (Secrets) and add `KRX_ID` and `KRX_PW` (your free data.krx.co.kr
login) with notebook access on. Never paste them into a cell or into chat. If they are missing, the notebook tries
without a login and says so. Then Runtime → Run all. The download takes about 20–40 minutes.
Send back the final printed JSON.

Calibration code: `p1ks_dev.py`, SHA-256 `33cb99b624feb013fdf2116103aa87eb93a274aa816a07f43329034ae2a833a2`.

In [ ]:
import os, subprocess, sys, time, json, hashlib, random
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pykrx==1.2.9'], check=True)
from google.colab import userdata
missing = []
for k in ('KRX_ID', 'KRX_PW'):
    try:
        os.environ[k] = userdata.get(k)
    except Exception as e:
        missing.append(k + ' (' + type(e).__name__ + ')')
if missing:
    print('KRX secrets not available:', ', '.join(missing))
    print('Continuing without a KRX login. If the download step fails, add the secrets (key icon) and rerun.')
else:
    print('KRX login secrets found.')
SRC = '"""P1KS development-window calibration (2015-2019 only): summary statistics for the power check.\n\nInput: for each stock, a daily DataFrame with columns\n    close   adjusted close,\n    mcap    market capitalization,\n    fnet    foreign net purchase value (KRX categories 외국인 + 기타외국인),\nindexed by a DatetimeIndex. Prices may start 2014-09-01; every date must be before 2020-01-01.\n\nPer stock and window (full 2015-2019, and halves split at 2017-07-01):\n    x_t   log close minus its trailing 60-day mean, divided by the stock\'s s.d. of daily log returns (2015-2019),\n    u_t   foreign net purchase / market cap at t-1, winsorized at the window\'s 0.5% and 99.5% quantiles,\n    tagged rule  u_t = alpha + r u_{t-1} + w x_{t-1} + e            (OLS),\n    impact       x_t = const + sum_{j<=5} a_j x_{t-j} + kappa u_{t-1} + e   (OLS),\n    c            kappa * w / (1 - r)   (foreign budget contribution; requires |r| < 0.99),\n    S_hat        sum of coefficients of an AR(10) in x with constant (prices only).\nOutput: cross-stock summaries only (no stock-level values).\n"""\nimport numpy as np\nimport pandas as pd\n\nCUTOFF = pd.Timestamp(\'2020-01-01\')\nSTART = pd.Timestamp(\'2015-01-01\')\nSPLIT = pd.Timestamp(\'2017-07-01\')\nAR_LAGS = 10\nIMPACT_LAGS = 5\nMIN_DAYS = 200\n\n\ndef _ols(y, X):\n    X = np.column_stack([np.ones(len(y)), X])\n    beta, *_ = np.linalg.lstsq(X, y, rcond=None)\n    return beta[1:]\n\n\ndef _lags(v, k):\n    n = len(v)\n    return np.column_stack([v[k - j:n - j] for j in range(1, k + 1)])\n\n\ndef prepare(df):\n    df = df.sort_index()\n    assert df.index.max() < CUTOFF, \'data after 2019 must not be read\'\n    lc = np.log(df[\'close\'].astype(float))\n    ret = lc.diff()\n    sd = ret[ret.index >= START].std()\n    x = (lc - lc.rolling(60).mean()) / sd\n    u = df[\'fnet\'].astype(float) / df[\'mcap\'].astype(float).shift(1)\n    out = pd.DataFrame({\'x\': x, \'u\': u}).replace([np.inf, -np.inf], np.nan)\n    return out[out.index >= START].dropna()\n\n\ndef window_stats(d):\n    if len(d) < MIN_DAYS:\n        return None\n    u = d[\'u\'].to_numpy().copy()\n    lo, hi = np.quantile(u, [0.005, 0.995])\n    u = np.clip(u, lo, hi)\n    x = d[\'x\'].to_numpy()\n    if np.std(u) == 0 or np.std(x) == 0:\n        return None\n    r, w = _ols(u[1:], np.column_stack([u[:-1], x[:-1]]))\n    k = IMPACT_LAGS\n    Xi = np.column_stack([_lags(x, k), u[k - 1:-1]])\n    coef = _ols(x[k:], Xi)\n    kappa = coef[-1]\n    ar = _ols(x[AR_LAGS:], _lags(x, AR_LAGS))\n    s_hat = float(np.sum(ar))\n    c = float(kappa * w / (1 - r)) if abs(r) < 0.99 else np.nan\n    return {\'r\': float(r), \'w\': float(w), \'kappa\': float(kappa), \'c\': c, \'S\': s_hat, \'n\': int(len(d))}\n\n\ndef stock_stats(df):\n    d = prepare(df)\n    return {\'full\': window_stats(d), \'h1\': window_stats(d[d.index < SPLIT]), \'h2\': window_stats(d[d.index >= SPLIT])}\n\n\ndef _q(v):\n    v = np.asarray([a for a in v if np.isfinite(a)])\n    if len(v) == 0:\n        return None\n    qs = np.quantile(v, [0.05, 0.25, 0.5, 0.75, 0.95])\n    return {\'n\': int(len(v)), \'mean\': float(v.mean()), \'sd\': float(v.std(ddof=1)) if len(v) > 1 else None,\n            \'q05\': float(qs[0]), \'q25\': float(qs[1]), \'median\': float(qs[2]), \'q75\': float(qs[3]), \'q95\': float(qs[4])}\n\n\ndef summarize(results, groups):\n    """results: ticker -> stock_stats output; groups: ticker -> \'index\' or \'other\'."""\n    out = {\'n_stocks_input\': len(results)}\n    for g in (\'index\', \'other\', \'all\'):\n        tick = [t for t in results if g == \'all\' or groups.get(t) == g]\n        full = [results[t][\'full\'] for t in tick if results[t][\'full\']]\n        both = [(results[t][\'h1\'], results[t][\'h2\']) for t in tick\n                if results[t][\'h1\'] and results[t][\'h2\']\n                and np.isfinite(results[t][\'h1\'][\'c\']) and np.isfinite(results[t][\'h2\'][\'c\'])]\n        s = {\'n_full\': len(full), \'n_both_halves\': len(both)}\n        s[\'c_full\'] = _q([f[\'c\'] for f in full])\n        s[\'r_full\'] = _q([f[\'r\'] for f in full])\n        s[\'w_full\'] = _q([f[\'w\'] for f in full])\n        s[\'kappa_full\'] = _q([f[\'kappa\'] for f in full])\n        s[\'S_full\'] = _q([f[\'S\'] for f in full])\n        s[\'days_full\'] = _q([f[\'n\'] for f in full])\n        if len(both) > 5:\n            c1 = np.array([a[\'c\'] for a, _ in both]); c2 = np.array([b[\'c\'] for _, b in both])\n            S1 = np.array([a[\'S\'] for a, _ in both]); S2 = np.array([b[\'S\'] for _, b in both])\n            dS, dc = S2 - S1, c2 - c1\n            s[\'dS_placebo\'] = _q(dS)\n            s[\'dc_placebo\'] = _q(dc)\n            s[\'corr_dS_dc\'] = float(np.corrcoef(dS, dc)[0, 1])\n            s[\'corr_c_h1_h2\'] = float(np.corrcoef(c1, c2)[0, 1])\n            s[\'spearman_c_h1_h2\'] = float(pd.Series(c1).rank().corr(pd.Series(c2).rank()))\n            s[\'days_h1\'] = _q([a[\'n\'] for a, _ in both]); s[\'days_h2\'] = _q([b[\'n\'] for _, b in both])\n        out[g] = s\n    return out\n'
assert hashlib.sha256(SRC.encode()).hexdigest() == '33cb99b624feb013fdf2116103aa87eb93a274aa816a07f43329034ae2a833a2', 'calibration code changed'
open('p1ks_dev.py', 'w').write(SRC)
print('setup ok')

In [ ]:
import time, json, hashlib, random
import numpy as np, pandas as pd
from pykrx import stock
import p1ks_dev as P

DAY = '20190102'
P_START, F_START, END = '20140901', '20150102', '20191230'
assert END < '20200101'
idx = set(stock.get_index_portfolio_deposit_file('1028', DAY)) | set(stock.get_index_portfolio_deposit_file('2203', DAY))
allt = sorted(t for m in ('KOSPI', 'KOSDAQ') for t in stock.get_market_ticker_list(DAY, market=m) if t.endswith('0'))
others = sorted(t for t in allt if t not in idx)
rng = random.Random(20261127)
sample_other = sorted(rng.sample(others, min(400, len(others))))
universe = sorted(idx) + sample_other
groups = {t: ('index' if t in idx else 'other') for t in universe}
print('index constituents:', len(idx), ' other sampled:', len(sample_other), ' of', len(others))

def fetch(t):
    for attempt in range(4):
        try:
            px = stock.get_market_ohlcv_by_date(P_START, END, t, adjusted=True)
            time.sleep(0.25)
            cap = stock.get_market_cap_by_date(P_START, END, t)
            time.sleep(0.25)
            fl = stock.get_market_trading_value_by_date(F_START, END, t, detail=True)
            time.sleep(0.25)
            fcols = [c for c in fl.columns if c in ('외국인', '기타외국인')]
            if len(fcols) != 2:
                return None, 'foreign columns: ' + ','.join(map(str, fl.columns))
            df = pd.DataFrame({'close': px['종가'], 'mcap': cap['시가총액']}).join(fl[fcols].sum(axis=1).rename('fnet'))
            df.index = pd.to_datetime(df.index)
            df = df[(df.index < pd.Timestamp('2020-01-01'))]
            df['fnet'] = df['fnet'].where(df.index >= pd.Timestamp('2015-01-02'))
            return df, None
        except Exception as e:
            err = type(e).__name__ + ': ' + str(e)[:120]
            time.sleep(2 + 3 * attempt)
    return None, err

probe, perr = fetch(universe[0])
if probe is None:
    raise SystemExit('KRX download failed for the first stock: ' + str(perr) + '\nAdd the KRX_ID / KRX_PW secrets and rerun.')
print('first download ok:', len(probe), 'rows')

results, failures = {}, {}
for i, t in enumerate(universe):
    df, err = fetch(t)
    if df is None:
        failures[t] = err
        continue
    try:
        results[t] = P.stock_stats(df)
    except Exception as e:
        failures[t] = 'estimate: ' + type(e).__name__
    if i % 50 == 0:
        print(i, 'of', len(universe), 'done;', len(failures), 'failures', flush=True)

summary = P.summarize(results, groups)
summary['n_failures'] = len(failures)
summary['failure_kinds'] = sorted(set(v.split(':')[0] for v in failures.values()))[:10]
summary['code_sha256'] = hashlib.sha256(open('p1ks_dev.py').read().encode()).hexdigest()
print(json.dumps(summary, indent=1, ensure_ascii=False))
print('Only cross-stock summaries were printed. No data dated 2020 or later were read.')